# CodeOfConduct-Agents Notebook Summary

## Overview

This notebook uses CrewAI agents to automate the process of discovering, analyzing, and comparing corporate Supplier Codes of Conduct. It aims to identify key policy areas, extract relevant information, and assess similarities between different companies' policies.

## Workflow

The notebook follows these main steps:

1. **Configuration Setup:** Installs necessary libraries, defines API keys, and sets environment variables.
2. **Constants Definition:** Defines key constants like the default model, mission statement, and policy disciplines.
3. **Imports and Utility Functions:** Imports necessary modules and defines reusable functions for file operations and URL handling.
4. **Dynamic Discovery of Code of Conduct Policies:**
    - Uses a **Discovery Agent** to find potential URLs related to supplier conduct policies.
    - Employs a **Validation Agent** to summarize and validate the discovered URLs, ensuring relevance.
    - Saves the validated URLs to files for each company.
5. **Extract Policy Content From Companies Validated Web Resources:**
    - Creates specialized **Discipline Agents** to analyze specific policy areas within the Code of Conduct documents.
    - Summarizes each company's stance on various disciplines, citing sources.
    - Stores the analysis results in separate files for each discipline and company.
6. **Compare Company Code of Conduct Policies:**
    - Creates a **Comparison Agent** to compare two companies' policies for each discipline.
    - Generates comparison reports highlighting similarities, differences, and assigning similarity scores.
    - Saves the comparison reports to files.
7. **Create Report:**
    - Creates a **Summary Report Agent** to generate a succinct overview of the comparison results.
    - Compiles the key findings, similarity scores, and summaries into a final report file.
8. **Sample Execution:** Provides an example of how to run the entire workflow using two companies, "Amazon" and "UPS."

## Key Features

- Leverages CrewAI agents and tools for automation and efficient policy analysis.
- Focuses on specific policy disciplines for detailed comparison.
- Provides clear and concise summaries of company stances and comparison results.
- Generates reports that can be used for compliance evaluations and procurement decisions

## Configuration Setup
Installs CrewAI and associated tools. Defines API keys loading from Colab's secret store, and sets environment variables needed by CrewAI and associated tools.

In [ ]:
%%capture --no-stderr
%pip install -U --quiet 'crewai[tools]' aisuite databricks-sdk

In [ ]:
# Constants and API Key Configuration
import os
from google.colab import userdata

# === Load API keys securely from Google Colab Secrets ===
def load_api_keys():
    keys = {
        "HUGGINGFACEHUB_API_TOKEN": userdata.get("HUGGINGFACEHUB_ACCESS_TOKEN"),
        "SERPER_API_KEY": userdata.get("SERPER_API_KEY"),
        "OPENAI_API_KEY": userdata.get("OPENAI_API_KEY"),
        "GEMINI_API_KEY": userdata.get("GEMINI_API_KEY"),
    }
    for key, value in keys.items():
        if not value:
            raise ValueError(f"❌ Missing {key}. Please set this API key in Colab secrets.")
        os.environ[key] = value
    print("✅ All API keys loaded and configured successfully.")

# Execute API key loading upon running this cell
load_api_keys()

## Constants Definition

Sets key constants including the default model configuration and the list of policy disciplines. A central MISSION constant, clearly articulating the overall objective that guides all agents. These constants are used globally across other cells, ensuring easy maintenance and consistency.

In [ ]:
# === Model Configuration ===
DEFAULT_MODEL = "gpt-4o-mini"

# === Mission Statement ===
MISSION = (
    "Identify, validate, extract, and analyze corporate Supplier Codes of Conduct "
    "and related policies to enable accurate comparisons across suppliers, "
    "support compliance evaluations, and drive informed procurement decisions."
)

# === Policy Disciplines ===
POLICY_DISCIPLINES = [
    "Environmental Protection",
    "Child Labor",
    "Forced Labor and Prison Labor",
    "Non-Discrimination and Equal Opportunity",
    "Anti-Bribery and Anti-Corruption",
    "Responsible Sourcing of Materials",
    "Health and Safety",
    "Wages Benefits and Working Hours",
    "Monitoring Auditing and Compliance",
    "Compliance with Local and International Laws",
    "Prohibition of Human Trafficking and Modern Slavery",
    "Data Privacy and Intellectual Property Protection"
]

## Imports and Utility Functions
Defines reusable utility functions to handle file operations

In [ ]:
# === Import necessary modules ===
import os
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, WebsiteSearchTool, FileReadTool


# === Instantiate Tools ===
search_tool = SerperDevTool()           # For general web search
web_rag_tool = WebsiteSearchTool()      # For content summarization and validation
file_read_tool = FileReadTool()         # For reading files

# === Save URLs to File ===
def save_urls_to_file(filename, urls):
    """Save the validated URLs to a file."""
    with open(filename, 'w') as file:
        if urls:
            file.write("\n".join(urls))
        else:
            file.write("nil")
    print(f"✅ Saved validated URLs to {filename}")

# === Load URL from Company File ===
def load_company_url(company_name):
    """Load the first valid URL from the company file."""
    filename = f"{company_name.replace(' ', '_')}-CoC-URLS.txt"
    if os.path.exists(filename):
        with open(filename, 'r') as file:
            for line in file:
                line = line.strip()
                if line and line.lower() != "nil":
                    return line
    return None


# Dynamic Discovery of Code of Conduct Policies

In [ ]:

# === Discovery Agent ===
discovery_agent = Agent(
    role="Discovery Agent",
    goal="Discover potential URLs related to Supplier Code of Conduct, Supply Chain Standards, and Supplier Trust Policies.",
    backstory="An expert in sourcing credible and relevant URLs from the web, specifically focused on company supply chain policies and supplier conduct guidelines.",
    tools=[search_tool],
    llm=DEFAULT_MODEL,
    verbose=True,
)

# === Validation Agent ===
validation_agent = Agent(
    role="Validation Agent",
    goal="Validate and summarize URLs, ensuring they relate to the company's Supplier Code of Conduct or Supply Chain Trust Policies.",
    backstory="An expert in analyzing and summarizing web content to confirm relevance to the correct company and ensure detailed compliance information.",
    tools=[web_rag_tool],
    llm=DEFAULT_MODEL,
    verbose=True,
)

# === Main Discovery and Validation Function ===
def discover_validate_and_save_urls(company_name, url_count=3):
    """Discover and validate the most relevant URLs for a company."""

    # Define the search terms
    search_terms = (
        f'"{company_name}" "Supplier Code of Conduct" OR '
        f'"{company_name}" "Supply Chain Standards" OR '
        f'"{company_name}" "Supplier Trust Policy"'
    )

    # === Discovery Task ===
    discovery_description = (
        f"Perform a detailed web search using the query: '{search_terms}'.\n"
        "- Focus on publicly accessible documents that detail **supplier conduct, supply chain standards, or supplier trust policies**.\n"
        "- Exclude irrelevant content such as **employee conduct, business ethics for employees, or general marketing materials**.\n"
        "- Provide up to {url_count} of the **most relevant URLs**."
    ).format(url_count=url_count)

    discovery_task = Task(
        description=discovery_description,
        expected_output=f"A list of up to {url_count} relevant and accessible URLs for the company '{company_name}'.",
        agent=discovery_agent
    )

    # === Validation Task ===
    validation_description = (
        "Summarize the content of each discovered URL and validate the following:\n"
        "- Ensure the **title or content** explicitly mentions the company name '{company_name}' to confirm relevance.\n"
        "- Confirm the document focuses on **supplier conduct, supply chain standards, or supplier trust policies**.\n"
        "- Exclude documents related to **employee conduct**, general business ethics, or marketing pages.\n"
        "- Clearly reason why the content is relevant or not.\n"
        "- Provide a summarized content description and **state if the URL is valid or invalid** based on its focus on supply chain trust."
    )

    validation_task = Task(
        description=validation_description,
        expected_output="A filtered list of URLs deemed valid, with reasoning for inclusion or exclusion.",
        agent=validation_agent,
        context=[discovery_task]
    )

    # === Create and Run the Crew ===
    discovery_validation_crew = Crew(
        agents=[discovery_agent, validation_agent],
        tasks=[discovery_task, validation_task],
        verbose=True,
    )

    discovery_validation_crew.kickoff()

    # === Process Results ===
    validated_urls = validation_task.output.raw.strip().splitlines()[:url_count]

    # Print to console for review
    print(f"\n🔍 Validated URLs for {company_name}:\n{validated_urls if validated_urls else 'No valid URLs found.'}")

    # === Save to File ===
    filename = f"{company_name.replace(' ', '_')}-CoC-URLS.txt"
    save_urls_to_file(filename, validated_urls)

# === Example Execution ===
# discover_validate_and_save_urls("Amazon", url_count=3)
# discover_validate_and_save_urls("UPS", url_count=3)


# Extract Policy Content From Companies Validated Web Resources

In [ ]:
# === Create Specialized Discipline Agents ===
def create_discipline_agent(discipline, company_name):
    """Create an agent for a specific Code of Conduct discipline with self-reflection."""
    return Agent(
        role=f"{discipline} Analysis Agent",
        goal=(
            f"Summarize {company_name}'s stance on {discipline} from their Supplier Code of Conduct."
        ),
        backstory=(
            f"You are an expert in compliance policy analysis. Summarize {discipline} accurately based only on the provided document. "
            "If not mentioned, clearly state 'N/A'. Include the source(s) at the end of your summary."
        ),
        tools=[web_rag_tool],
        llm=DEFAULT_MODEL,
        verbose=True,
    )

# === Main Analysis Function ===
def analyze_company_code_of_conduct(company_name):
    """Analyze a company's Code of Conduct based on key disciplines."""

    url = load_company_url(company_name)
    if not url:
        print(f"⚠️ No valid URL found for {company_name}. Skipping analysis.")
        return

    # List of Code of Conduct Disciplines
    disciplines = POLICY_DISCIPLINES

    # Create directory for company outputs
    output_directory = f"{company_name.replace(' ', '_')}"
    os.makedirs(output_directory, exist_ok=True)

    agents = [create_discipline_agent(discipline, company_name) for discipline in disciplines]

    # Create and configure tasks with proper `output_file` usage
    tasks = [
        Task(
            description=(
                f"Analyze the document at '{url}' and summarize {company_name}'s stance on **{discipline}**.\n"
                "- If explicitly mentioned, summarize the policy or stance clearly and concisely.\n"
                "- If not mentioned, state 'N/A'. Avoid assumptions or made-up details.\n"
                "- If best practices are encouraged, provide brief examples if mentioned.\n"
                "- Mention how the company monitors or enforces compliance, if stated.\n"
                "- Include the source URL at the end of the summary."
            ),
            expected_output=(
                f"A summary of the company's position on {discipline}, or 'N/A' if not addressed. "
                "Cite the source(s) at the end of the summary."
            ),
            agent=agents[i],
            output_file=f"{output_directory}/{discipline.replace(' ', '_')}.txt"
        ) for i, discipline in enumerate(disciplines)
    ]

    # === Create and Run the Crew ===
    analysis_crew = Crew(
        agents=agents,
        tasks=tasks,
        verbose=True,
    )

    analysis_crew.kickoff()

    print(f"\n✅ Analysis completed. Check '{output_directory}/' for detailed discipline summaries.")

# === Example Execution ===
analyze_company_code_of_conduct("UPS")


# Compare Company Code of Conduct Policies

In [ ]:

# === Create Comparison Agent ===
def create_comparison_agent(policies):
    """Create an agent to compare two company policies for a policy area."""
    return Agent(
        role="Policy Comparison Agent",
        goal="Compare two companies' policy stances and generate similarity insights.",
        backstory=(
            "You are an expert in policy comparison. Analyze the content from two companies and summarize key similarities and differences. "
            "Assign a similarity score (0 to 100) based on alignment, and justify your assessment."
        ),
        tools=[web_rag_tool, file_read_tool],
        llm=DEFAULT_MODEL,
        verbose=True,
    )

# === Main Comparison Function ===
def compare_companies(base_company, comparison_company):
    """Compare two companies' policy disciplines."""

    # List of Policy Disciplines
    disciplines = POLICY_DISCIPLINES

    # Create output directory for comparison results
    comparison_dir = f"{base_company.replace(' ', '_')}/Comparison-Reports/{comparison_company.replace(' ', '_')}"
    os.makedirs(comparison_dir, exist_ok=True)

    agents = [create_comparison_agent(discipline) for discipline in disciplines]

    tasks = []

    for i, discipline in enumerate(disciplines):
        base_file = f"{base_company.replace(' ', '_')}/{discipline.replace(' ', '_')}.txt"
        compare_file = f"{comparison_company.replace(' ', '_')}/{discipline.replace(' ', '_')}.txt"
        comparison_file = f"{comparison_dir}/{discipline.replace(' ', '_')}-Comparison.txt"

        task = Task(
            description=(
                f"Compare the following policy summaries for the discipline '{discipline}'.\n\n"
                f"1. **Base Company ({base_company}) Policy:** (Content from file: '{base_file}')\n"
                f"2. **Comparison Company ({comparison_company}) Policy:** (Content from file: '{compare_file}')\n\n"
                "- Summarize key **similarities and differences**.\n"
                "- Provide a **similarity score (0-100)** reflecting how closely aligned the policies are.\n"
                "- Clearly **justify** the similarity score.\n"
                "- If a policy is missing (N/A), clearly state so and reflect it in the score."
            ),
            expected_output=(
                f"A detailed comparison summary for {discipline}, including:\n"
                "- Similarities\n"
                "- Differences\n"
                "- Similarity score (0-100) with justification."
            ),
            agent=agents[i],
            output_file=comparison_file,
            tools=[file_read_tool],
        )

        tasks.append(task)

    # === Create and Run the Comparison Crew ===
    comparison_crew = Crew(
        agents=agents,
        tasks=tasks,
        verbose=True,
    )

    comparison_crew.kickoff()

    print(f"\n✅ Comparison completed. Check '{comparison_dir}/' for detailed reports.")

# === Example Execution ===
compare_companies("Amazon", "UPS")


# Create Report

In [ ]:
# === Create Summary Agent ===
def create_summary_agent(file_path):
    """Create an agent with direct file path access."""
    return Agent(
        role="Summary Report Agent",
        goal="Generate a succinct report summarizing policy similarities, differences, and similarity scores.",
        backstory=(
            "You are an expert at summarizing policy comparisons. Review the given comparison content, "
            "extract key points, and generate a clear summary with accurate similarity scores."
        ),
        tools=[FileReadTool(file_path=file_path)],
        llm=DEFAULT_MODEL,
        verbose=True,
    )

# === Function to Generate Summary Report ===
def generate_summary_report(company_one, company_two):
    """Generate a succinct summary report comparing two companies."""

    # Define directories
    base_dir = company_one.replace(' ', '_')
    comparison_dir = os.path.join(base_dir, "Comparison-Reports", company_two.replace(' ', '_'))
    output_file = os.path.join(base_dir, "Comparison-Reports", f"{company_two.replace(' ', '_')}-Summary.txt")

    if not os.path.exists(comparison_dir):
        print(f"⚠️ No comparison reports found for {company_two} in {comparison_dir}.")
        return

    # Collect comparison file paths dynamically
    comparison_files = [
        os.path.join(comparison_dir, f) for f in os.listdir(comparison_dir)
        if f.endswith("-Comparison.txt")
    ]

    if not comparison_files:
        print(f"⚠️ No comparison files found in {comparison_dir}.")
        return

    combined_content = ""

    tasks = []

    # Create a summarization task for each file
    for file_path in comparison_files:
        discipline = os.path.basename(file_path).replace("-Comparison.txt", "").replace("_", " ")

        agent = create_summary_agent(file_path)

        task = Task(
            description=(
                f"Review the comparison report for the discipline **{discipline}**.\n\n"
                "- Provide a **short summary** of key similarities and differences strictly from the file content.\n"
                "- Include the **similarity score** exactly as found in the file.\n"
                "- Avoid embellishing or assuming details not present in the file."
            ),
            expected_output=(
                "A structured section that includes:\n"
                "- Policy Discipline\n"
                "- Similarities and differences summary\n"
                "- Similarity score"
            ),
            agent=agent
        )

        tasks.append(task)

    # Create a summarization crew
    summary_crew = Crew(
        agents=[t.agent for t in tasks],
        tasks=tasks,
        verbose=True,
    )

    summary_crew.kickoff()

    # Collect and write final summary report
    with open(output_file, 'w') as final_report:
        for task in tasks:
            final_report.write(f"{task.output.raw.strip()}\n\n")

    print(f"\n✅ Summary report generated: '{output_file}'")

# === Example Execution ===
generate_summary_report("Amazon", "UPS")

# Sample Execution

In [ ]:

base_company = "Amazon"
comparison_company = "UPS"

# Discover and validate URLs related to Code of Conduct Policies

discover_validate_and_save_urls(base_company, url_count=3)
discover_validate_and_save_urls(comparison_company, url_count=3)

# Extract and Analyze Code of Conduct Across Policies
analyze_company_code_of_conduct(base_company)
analyze_company_code_of_conduct(comparison_company)

# Compare Companies CoC Policy with Similarity Assessment
compare_companies(base_company, comparison_company

# === Create Report ===
generate_summary_report(base_company, comparison_company)